## 1. Data Preprocessing
In this section, we import the raw ESS data and prepare it for analysis by handling missing values and transforming variables.

In [7]:
import pandas as pd
import numpy as np
import zipfile
import matplotlib.pyplot as plt
import seaborn as sns

# Load Dataset
zip_path = "ESS9e03_3-ESS10e03_3-ESS10SCe03_2-ESS11e04_1-subset.zip"
csv_inside = "ESS9e03_3-ESS10e03_3-ESS10SCe03_2-ESS11e04_1-subset.csv"

with zipfile.ZipFile(zip_path, 'r') as z:
    with z.open(csv_inside) as f:
        df = pd.read_csv(f, low_memory=False)

print("CSV Load Successful!")
print(f"Dataset Size: {df.shape[0]} rows x {df.shape[1]} columns")

CSV Load Successful!
Dataset Size: 159320 rows x 635 columns


### Data Exploration
Before cleaning, we examine the raw structure of the data, including country distribution and the default scaling of the human value variables.

In [8]:
# Quick preview of key variables
interest_cols = ['cntry', 'yrbrn'] + [c for c in df.columns if c.startswith('ip')][:5]
display(df[interest_cols].head())

print("\n### Country Distribution (Top 10)")
display(df['cntry'].value_counts().head(10))

,cntry,yrbrn,ipadvnt,ipadvnta,ipbhprp,ipbhprpa,ipcrtiv
0,AT,1975,2.0,NaN,2.0,NaN,2.0
1,AT,1951,5.0,NaN,2.0,NaN,3.0
2,AT,1978,3.0,NaN,2.0,NaN,3.0
3,AT,1955,6.0,NaN,1.0,NaN,2.0
4,AT,1947,6.0,NaN,1.0,NaN,2.0



### Country Distribution (Top 10)


cntry
DE    13503
IT     8250
BG     7155
AT     6856
IE     6003
ES     5795
FR     5758
HU     5628
GR     5556
RS     5111
Name: count, dtype: int64

### Data Cleansing & Variable Transformation
To ensure the data is research-ready, we perform the following:
1. **Missing Values:** Convert ESS codes (7, 8, 9, 77, 88, 99) to `NaN`.
2. **Scale Reversal:** We transform the 1–6 scale to ensure **6 = Strong Agreement**.
3. **Generational Mapping:** Creating cohorts (Gen Z to Boomers) based on `yrbrn`.
4. **Schwartz Dimensions:** Mapping 21 items into the 10 core values.

In [12]:
# 1. Handle Missing Data
value_cols = [c for c in df.columns if c.startswith('ip')]
df[value_cols] = df[value_cols].replace([7, 8, 9], np.nan)
df['lrscale'] = df['lrscale'].replace([77, 88, 99], np.nan)
df['yrbrn'] = df['yrbrn'].replace([9999], np.nan)

# 2. Reverse Coding (7 - x)
df[value_cols] = 7 - df[value_cols]

# 3. Create Generation Column
conditions = [
    (df['yrbrn'] >= 1997),
    (df['yrbrn'] >= 1981) & (df['yrbrn'] <= 1996),
    (df['yrbrn'] >= 1965) & (df['yrbrn'] <= 1980),
    (df['yrbrn'] <= 1964)
]
labels = ['Gen Z', 'Millennials', 'Gen X', 'Boomers']
df['generation'] = np.select(conditions, labels, default='Unknown')

# 4. Create Schwartz Value Dimensions (The Full 10)
# Re-mapped specifically to our dataset's column names

# Universalism: Understanding, Helping, Environment
df['Universalism'] = df[['ipudrst', 'iphlppl', 'impenv']].mean(axis=1)

# Benevolence: Helping, Loyalty to friends
df['Benevolence'] = df[['iphlppl', 'iplylfr']].mean(axis=1)

# Tradition: Modesty, Tradition
df['Tradition'] = df[['imptrad', 'ipmodst']].mean(axis=1)

# Security: Safety, Strong Government
df['Security'] = df[['impsafe', 'ipstrgv']].mean(axis=1)

# Conformity: Following rules, Behaving properly (your 'ipbhvar' replacement)
df['Conformity'] = df[['ipfrule', 'ipbhprp']].mean(axis=1)

# Self-Direction: Freedom, Creativity
df['Self-Direction'] = df[['impfree', 'ipcrtiv']].mean(axis=1)

# Stimulation: Excitement, Adventure
df['Stimulation'] = df[['ipadvnt', 'impdiff']].mean(axis=1)

# Hedonism: Good times, Fun
df['Hedonism'] = df[['ipgdtim', 'impfun']].mean(axis=1)

# Achievement: Success, Ability/Equality
df['Achievement'] = df[['ipsuces', 'ipshabt', 'ipeqopt']].mean(axis=1)

# Power: Wealth, Respect
df['Power'] = df[['imprich', 'iprspot']].mean(axis=1)

# Updated Master List for Analysis
schwartz_values = [
    'Universalism', 'Benevolence', 'Tradition', 'Security', 'Conformity',
    'Self-Direction', 'Stimulation', 'Hedonism', 'Achievement', 'Power'
]

print("Data Cleansing Complete: All 10 dimensions mapped to local variable names.")

# 5. Cleaned Data Preview for the team
print("\n### 6. Cleaned Data Preview")
display(df[['generation', 'lrscale'] + schwartz_values].head())

Data Cleansing Complete: All 10 dimensions mapped to local variable names.

### 6. Cleaned Data Preview


,generation,lrscale,Universalism,Benevolence,Tradition,Security,Conformity,Self-Direction,Stimulation,Hedonism,Achievement,Power
0,Gen X,9.0,2.666667,2.0,1.5,2.0,2.0,1.5,2.0,2.5,2.000000,2.0
1,Boomers,5.0,3.000000,2.5,3.5,2.5,3.0,3.0,4.5,3.0,3.333333,3.5
2,Gen X,5.0,2.666667,2.5,3.0,3.0,2.5,2.5,3.5,3.5,3.000000,4.0
3,Boomers,NaN,1.000000,1.0,2.0,1.0,1.5,1.5,4.5,3.0,1.000000,3.0
4,Boomers,5.0,2.000000,2.0,2.0,2.0,1.5,2.0,5.0,2.5,3.666667,4.5


While running the cleaning cell we detected a naming discrepancy in the raw ESS dataset where the standard Environment variable (ipenvir) was missing.
So we performed a programmatic audit of the data index to identify the local variable names. Then corrected the mapping to impenv to align with the specific ESS Round in our subset this ensured the Universalism index is calculated with 100% accuracy according to the Schwartz Theory

In [9]:
print([c for c in df.columns if 'env' in c])

['impenv', 'impenva']


Still kept running into errors, the code cell below was put to see every variable related to the Schwartz values. This will give us the "Map" we need to fix the whole thing at once:

In [11]:
# This prints all columns starting with 'ip' or 'imp'
print("--- Potential Value Columns ---")
print([c for c in df.columns if c.startswith('ip') or c.startswith('imp')])

--- Potential Value Columns ---
['implvdm', 'impcntr', 'impdiff', 'impdiffa', 'impenv', 'impenva', 'impfree', 'impfreea', 'impfun', 'impfuna', 'imprich', 'impricha', 'impsafe', 'impsafea', 'imptrad', 'imptrada', 'ipadvnt', 'ipadvnta', 'ipbhprp', 'ipbhprpa', 'ipcrtiv', 'ipcrtiva', 'ipeqopt', 'ipeqopta', 'ipfrule', 'ipfrulea', 'ipgdtim', 'ipgdtima', 'iphlppl', 'iphlppla', 'iplylfr', 'iplylfra', 'ipmodst', 'ipmodsta', 'iprspot', 'iprspota', 'ipshabt', 'ipshabta', 'ipstrgv', 'ipstrgva', 'ipsuces', 'ipsucesa', 'ipudrst', 'ipudrsta']


## 2. Descriptive Analysis
Now that the data is clean, we will begin answering our primary research questions by looking at the averages across generations and regions.
We have successfully built the 'Data Engine' for the project. By cleaning the missing codes, reversing the scales for intuitive reading, and mapping the Schwartz values, we have moved from Raw Data to Research-Ready Insights. The dataset is now fully prepared for Sibahle's correlations and Taya’s generational comparisons.

In [ ]:
# Final check: Average Universalism by Generation
print("### Mean Universalism Score by Generation")
display(df.groupby('generation')['Universalism'].mean().sort_values(ascending=False))